In [1]:
import pandas as pd 
import numpy as np

In [2]:
ENDERECO_DADOS = 'https://www.ispdados.rj.gov.br/Arquivos/BaseDPEvolucaoMensalCisp.csv'

df_estelionato = pd.read_csv(ENDERECO_DADOS, sep=';', encoding='iso-8859-1')
print(df_estelionato.head())
# print(df_estelionato.describe())
# print(df_estelionato['regiao'].unique())


   cisp  mes   ano  mes_ano  aisp  risp           munic    mcirc   regiao  \
0     1    1  2003  2003m01     5     1  Rio de Janeiro  3304557  Capital   
1     4    1  2003  2003m01     5     1  Rio de Janeiro  3304557  Capital   
2     5    1  2003  2003m01     5     1  Rio de Janeiro  3304557  Capital   
3     6    1  2003  2003m01     1     1  Rio de Janeiro  3304557  Capital   
4     7    1  2003  2003m01     1     1  Rio de Janeiro  3304557  Capital   

   hom_doloso  ...  cmp  cmba  ameaca  pessoas_desaparecidas  \
0           0  ...  NaN   NaN      21                      2   
1           3  ...  NaN   NaN      15                      6   
2           3  ...  NaN   NaN      47                      2   
3           6  ...  NaN   NaN      26                      2   
4           4  ...  NaN   NaN      10                      1   

   encontro_cadaver  encontro_ossada  pol_militares_mortos_serv  \
0                 0                0                          0   
1                 

In [3]:
# Delimitar as variáveis cisp regiao munic roubo_veiculo
df_estelionato = df_estelionato[['cisp', 'regiao', 'munic', 'estelionato']]

# Corrigindo a Grande Niterói
correcao_acentuacao = df_estelionato['regiao'].str.startswith('Grande Niter', na=False)
# print(correcao_acentuacao)
df_estelionato.loc[correcao_acentuacao, 'regiao'] = 'Grande Niterói'
print(df_estelionato['regiao'].unique())

<StringArray>
['Capital', 'Baixada Fluminense', 'Interior', 'Grande Niterói']
Length: 4, dtype: str


In [4]:
# Agrupando e totalizando os roubos de veículo por cisp regiao e munic
df_estelionato = df_estelionato.groupby(['cisp', 'regiao', 'munic'], as_index=False)['estelionato'].sum()
display(df_estelionato)

,cisp,regiao,munic,estelionato
0,1,Capital,Rio de Janeiro,28249
1,4,Capital,Rio de Janeiro,7540
2,5,Capital,Rio de Janeiro,23396
3,6,Capital,Rio de Janeiro,8154
4,7,Capital,Rio de Janeiro,2695
...,...,...,...,...
142,165,Interior,Mangaratiba,2418
143,166,Interior,Angra dos Reis,9811
144,167,Capital,Paraty,133
145,167,Interior,Paraty,2653


In [6]:
# Calculando média, mediana, total, maior, menor e quartis
array_estelionato = np.array(df_estelionato['estelionato'])

media = np.mean(array_estelionato)
mediana = np.median(array_estelionato)

maximo = np.max(array_estelionato)
minimo = np.min(array_estelionato)
total = np.sum(array_estelionato)


q1 = np.percentile(array_estelionato, 25)
q3 = np.percentile(array_estelionato, 75)

print('Média: ', media)
print('Mediana: ', mediana)
print('Máximo: ', maximo)
print('Mínimo: ', minimo)
print('Total: ', total)

print('Quartil 1: ', q1)
print('Quartil 3: ', q3)

Média:  8567.3537414966
Mediana:  6180.0
Máximo:  47399
Mínimo:  7
Total:  1259401
Quartil 1:  1231.0
Quartil 3:  12601.5


In [7]:
# Criando uma série com os maiores
df_maiores = df_estelionato[df_estelionato['estelionato'] >= q3].copy()
df_maiores['flag'] = 'mais'
display(df_maiores)

# Criando uma série com os menores
df_menores = df_estelionato[df_estelionato['estelionato'] <= q1].copy()
df_menores['flag'] = 'menos'
display(df_menores)



,cisp,regiao,munic,estelionato,flag
0,1,Capital,Rio de Janeiro,28249,mais
2,5,Capital,Rio de Janeiro,23396,mais
5,9,Capital,Rio de Janeiro,19992,mais
6,10,Capital,Rio de Janeiro,21239,mais
8,12,Capital,Rio de Janeiro,25120,mais
9,13,Capital,Rio de Janeiro,12670,mais
10,14,Capital,Rio de Janeiro,21506,mais
12,16,Capital,Rio de Janeiro,47399,mais
15,19,Capital,Rio de Janeiro,23146,mais
16,20,Capital,Rio de Janeiro,15874,mais


,cisp,regiao,munic,estelionato,flag
41,45,Capital,Rio de Janeiro,98,menos
49,55,Baixada Fluminense,Queimados;Japeri,50,menos
60,65,Baixada Fluminense,Magé;Guapimirim,624,menos
63,70,Interior,Tanguá,868,menos
65,71,Interior,Itaboraí;Tanguá,1031,menos
80,92,Interior,Rio das Flores,337,menos
82,94,Interior,Piraí,1022,menos
83,94,Interior,Piraí;Pinheiral,8,menos
86,97,Interior,Mendes,963,menos
87,98,Interior,Engenheiro Paulo de Frontin,455,menos


In [8]:
# Concatenando os dois dataframes df_maiores e df_menores
df_estelionato_flags = pd.concat([df_maiores, df_menores], ignore_index=True)

In [36]:
# exportando os dados utf-8-sig p/ ler no powerbi
df_estelionato_flags.to_csv('estelionato_flags.csv', index=False, encoding='utf-8-sig')